In [ ]:
import pandas as pd
import numpy as np
import re
from google_play_scraper import Sort, reviews
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import RandomOverSampler


# STEP 1: SCRAPING DATA (Kriteria 1)

print("Sedang mengambil data dari Google Play Store...")
app_id = 'com.kurogame.wutheringwaves.global'

result, _ = reviews(
    app_id,
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=10000
)

df = pd.DataFrame(result)
df = df[['content', 'score']]
print(f"Total data yang didapat: {len(df)}")

norm_dict = {
    "yg": "yang", "gk": "tidak", "gak": "tidak", "ga": "tidak",
    "bgt": "banget", "udh": "sudah", "udah": "sudah", "uadh": "sudah",
    "jd": "jadi", "sdh": "sudah", "tp": "tapi", "tpi": "tapi",
    "kalo": "kalau", "lu": "kamu", "gw": "saya", "gue": "saya",
    "bagus": "mantap", "jelek": "buruk", "ampas": "buruk", "keren": "mantap"
}

def clean_text(text):
    text = str(text).lower()

    text = re.sub(r'(.)\1+', r'\1', text)
    text = re.sub(r'[^a-z\s]', '', text)

    words = text.split()
    text = " ".join([norm_dict.get(w, w) for w in words])
    return text.strip()

def label_sentiment(score):
    if score <= 2: return 0
    elif score == 3: return 1
    else: return 2

print("Melakukan Preprocessing dan Pelabelan...")
df['content_cleaned'] = df['content'].apply(clean_text)
df['label'] = df['score'].apply(label_sentiment)
df = df[df['content_cleaned'] != ""].drop_duplicates(subset=['content_cleaned'])

X_raw = df['content_cleaned']
y = df['label']


tfidf_optim = TfidfVectorizer(max_features=8000, ngram_range=(1, 3), min_df=5)
X_tfidf = tfidf_optim.fit_transform(X_raw)


ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_tfidf, y)


# --- SKEMA 1: SVM (RBF Kernel) + Split 80/20 ---
X_train1, X_test1, y_train1, y_test1 = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)
model1 = SVC(kernel='rbf', C=10)
model1.fit(X_train1, y_train1)
acc1 = accuracy_score(y_test1, model1.predict(X_test1))

# --- SKEMA 2: Random Forest + Split 80/20 ---
model2 = RandomForestClassifier(n_estimators=200, random_state=42)
model2.fit(X_train1, y_train1)
acc2 = accuracy_score(y_test1, model2.predict(X_test1))

# --- SKEMA 3: Deep Learning (MLP) + Split 70/30 ---
X_train3, X_test3, y_train3, y_test3 = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42)
model3 = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=500, early_stopping=True, random_state=42)
model3.fit(X_train3, y_train3)
acc3 = accuracy_score(y_test3, model3.predict(X_test3))


print("\n" + "="*35)
print("HASIL PERCOBAAN SETELAH OPTIMASI")
print("="*35)
print(f"Skema 1 (SVM RBF)      : {acc1*100:.2f}%")
print(f"Skema 2 (Random Forest): {acc2*100:.2f}%")
print(f"Skema 3 (MLP Deep L.)  : {acc3*100:.2f}%")
print("="*35)


def predict_review(text):
    cleaned = clean_text(text)
    vector = tfidf_optim.transform([cleaned])

    prediction = model1.predict(vector)[0]
    labels = {0: 'Negatif', 1: 'Netral', 2: 'Positif'}
    return labels[prediction]

print("\nUJI COBA INFERENSI:")
texts = ["Gamenya ampas banget, sering lag dan gacha buruk!",
         "Grafiknya mantap tapi gameplay biasa aja",
         "Wuthering Waves seru banget, grafisnya luar biasa!"]

for t in texts:
    print(f"Teks: {t} -> Prediksi: {predict_review(t)}")


df.to_csv('dataset_wuthering_waves_cleaned.csv', index=False)

Sedang mengambil data dari Google Play Store...
Total data yang didapat: 10000
Melakukan Preprocessing dan Pelabelan...

HASIL PERCOBAAN SETELAH OPTIMASI
Skema 1 (SVM RBF)      : 96.35%
Skema 2 (Random Forest): 96.43%
Skema 3 (MLP Deep L.)  : 93.20%

UJI COBA INFERENSI:
Teks: Gamenya ampas banget, sering lag dan gacha buruk! -> Prediksi: Negatif
Teks: Grafiknya mantap tapi gameplay biasa aja -> Prediksi: Positif
Teks: Wuthering Waves seru banget, grafisnya luar biasa! -> Prediksi: Positif


In [ ]:
from sklearn.metrics import classification_report
y_pred_final = model1.predict(X_test1)

print("--- LAPORAN EVALUASI DETAIL (Kriteria 4) ---")

print(classification_report(y_test1, y_pred_final,
                            target_names=['Negatif (0)', 'Netral (1)', 'Positif (2)']))

--- LAPORAN EVALUASI DETAIL (Kriteria 4) ---
              precision    recall  f1-score   support

 Negatif (0)       0.93      0.97      0.95      1207
  Netral (1)       0.99      1.00      0.99      1145
 Positif (2)       0.97      0.92      0.95      1236

    accuracy                           0.96      3588
   macro avg       0.96      0.96      0.96      3588
weighted avg       0.96      0.96      0.96      3588



In [ ]:
#inference model
print("=== BUKTI INFERENSI MODEL ===")
test_kalam = [
    "Grafiknya jelek banget, mending main game sebelah",
    "Biasa aja sih, nggak ada yang spesial",
    "Gamenya keren, gacha-nya juga lumayan ramah buat F2P"
]

for t in test_kalam:
    hasil = predict_review(t)
    print(f"Input: {t}")
    print(f"Hasil Sentimen: {hasil}")
    print("-" * 20)

=== BUKTI INFERENSI MODEL ===
Input: Grafiknya jelek banget, mending main game sebelah
Hasil Sentimen: Negatif
--------------------
Input: Biasa aja sih, nggak ada yang spesial
Hasil Sentimen: Positif
--------------------
Input: Gamenya keren, gacha-nya juga lumayan ramah buat F2P
Hasil Sentimen: Positif
--------------------


In [2]:
import sys
!{sys.executable} -m pip install google-play-scraper

# File: kode_scraping.py
from google_play_scraper import Sort, reviews
import pandas as pd

app_id = 'com.kurogame.wutheringwaves.global'
result, _ = reviews(
    app_id,
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=10000
)

df = pd.DataFrame(result)
df.to_csv('dataset_wuthering_waves.csv', index=False)
print(f"Dataset berhasil disimpan. Total: {len(df)} baris.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.5 MB/s eta 0:00:00
Dataset berhasil disimpan. Total: 10000 baris.
